# Notebook 05: Sensitivity Analysis and Cross-Asset Validation

This notebook investigates how the number of hidden states K affects CHMM performance
(state-resolution sensitivity) and validates the model across multiple asset classes.
**No jumps** — CHMM only.

### Sections
1. **State-resolution sensitivity**: Sweep K in {3, 6, 9, 11, 13} and evaluate distributional metrics
2. **Sensitivity plots**: KS pass rate, ACF-MAE, and kurtosis vs K
3. **Cross-asset validation**: Train CHMM (K=6) on SPY, NVDA, JNJ, JPM and compare metrics

### Metrics
- **KS pass rate** (%) — Kolmogorov-Smirnov test at alpha=0.05
- **Excess kurtosis** — simulated vs observed
- **ACF-MAE** — mean absolute error of ACF(|Gt|) over lags 1-252
- **Wasserstein-1 distance** — mean earth-mover distance
- **Hellinger distance** — histogram overlap distance

## Setup

In [ ]:
include("../Include.jl");

## Configuration

In [ ]:
# --- TUNABLE PARAMETERS ---
K_VALUES = [3, 6, 9, 11, 13];
MAX_ITER = 60;
N_PATHS = 500;
L = 252;
risk_free_rate = 0.0;
Δt = 1/252;
primary_ticker = "SPY";

## Load Data

In [ ]:
# --- LOAD IN-SAMPLE DATA ---
train_dataset = MyPortfolioDataSet() |> x -> x["dataset"];
maximum_number_trading_days = nrow(train_dataset["AAPL"]);

dataset = Dict{String,DataFrame}();
for (t, data) in train_dataset
    if nrow(data) == maximum_number_trading_days
        dataset[t] = data;
    end
end

list_of_all_tickers = keys(dataset) |> collect |> sort;
all_firms_excess_return_matrix = log_growth_matrix(dataset, list_of_all_tickers; Δt=Δt, risk_free_rate=risk_free_rate);
idx_primary = findfirst(x -> x == primary_ticker, list_of_all_tickers);
R_is = all_firms_excess_return_matrix[:, idx_primary];

println("IS data: $(length(R_is)) observations for $primary_ticker")

In [ ]:
# --- LOAD OUT-OF-SAMPLE DATA ---
oos_dataset_raw = MyOutOfSamplePortfolioDataSet() |> x -> x["dataset"];
R_oos = log_growth_matrix(oos_dataset_raw, primary_ticker; Δt=Δt, risk_free_rate=risk_free_rate);

println("OoS data: $(length(R_oos)) observations for $primary_ticker")

## Evaluation Metrics Function

In [ ]:
# --- EVALUATION METRICS ---
function eval_metrics(observed::Vector{Float64}, simulated_archive::Matrix{Float64};
                      α=0.05, L_acf=252)

    n_paths = size(simulated_archive, 2);
    n_obs = length(observed);

    # Observed targets
    μ_obs = mean(observed); σ_obs = std(observed);
    kurt_obs = sum(((observed .- μ_obs) ./ σ_obs).^4) / n_obs - 3.0;
    τ_range = 1:min(L_acf, n_obs - 1);
    acf_obs = autocor(abs.(observed), τ_range);

    # Accumulators
    ks_pass = 0;
    kurt_sum = 0.0;
    acf_mae_sum = 0.0;
    w1_sum = 0.0;
    hell_sum = 0.0;

    for i in 1:n_paths
        sim = simulated_archive[:, i];

        # KS test
        ks_pval = pvalue(ApproximateTwoSampleKSTest(observed, sim));
        if ks_pval > α; ks_pass += 1; end

        # Kurtosis
        μ_s = mean(sim); σ_s = std(sim);
        kurt_sum += sum(((sim .- μ_s) ./ σ_s).^4) / length(sim) - 3.0;

        # ACF-MAE
        acf_sim = autocor(abs.(sim), τ_range);
        acf_mae_sum += mean(abs.(acf_obs .- acf_sim));

        # Wasserstein-1 (sorted quantile difference)
        obs_sorted = sort(observed);
        sim_sorted = sort(sim);
        n_min = min(length(obs_sorted), length(sim_sorted));
        obs_q = [obs_sorted[round(Int, k * length(obs_sorted) / n_min)] for k in 1:n_min];
        sim_q = [sim_sorted[round(Int, k * length(sim_sorted) / n_min)] for k in 1:n_min];
        w1_sum += mean(abs.(obs_q .- sim_q));

        # Hellinger distance (histogram-based)
        edges = range(minimum([observed; sim]) - 10, maximum([observed; sim]) + 10, length=101);
        h_obs = fit(Histogram, observed, edges).weights ./ length(observed);
        h_sim = fit(Histogram, sim, edges).weights ./ length(sim);
        hell_sum += sqrt(sum((sqrt.(h_obs) .- sqrt.(h_sim)).^2)) / sqrt(2);
    end

    return (
        ks_rate     = 100.0 * ks_pass / n_paths,
        kurtosis    = kurt_sum / n_paths,
        kurt_obs    = kurt_obs,
        acf_mae     = acf_mae_sum / n_paths,
        wasserstein = w1_sum / n_paths,
        hellinger   = hell_sum / n_paths
    );
end

## State-Resolution Sensitivity Loop

For each K in `K_VALUES`, train a CHMM on IS data, simulate `N_PATHS` paths,
and compute the full metric suite.

In [ ]:
# --- SENSITIVITY LOOP ---
sensitivity_results = Dict{Int, NamedTuple}();
sensitivity_models = Dict{Int, MyContinuousHiddenMarkovModel}();
number_of_steps = length(R_is);

for K in K_VALUES
    println("\n" * "="^50);
    println("Training CHMM with K=$K states...");
    println("="^50);

    # Train
    model_k = build(MyContinuousHiddenMarkovModel, (
        observations = R_is,
        number_of_states = K,
        max_iter = MAX_ITER
    ));
    sensitivity_models[K] = model_k;

    # Compute stationary distribution
    T_mat = zeros(K, K);
    for i in 1:K; T_mat[i, :] = probs(model_k.transition[i]); end
    π_stat = (T_mat^1000)[1, :];
    start_dist = Categorical(π_stat);

    # Simulate
    decoded = Array{Float64,2}(undef, number_of_steps, N_PATHS);
    for p in 1:N_PATHS
        start_state = rand(start_dist);
        states = model_k(start_state, number_of_steps);
        for j in 1:number_of_steps
            decoded[j, p] = rand(model_k.emission[states[j]]);
        end
    end

    # Evaluate
    m = eval_metrics(R_is, decoded; L_acf=L);
    sensitivity_results[K] = m;

    println("  KS pass rate:  $(round(m.ks_rate, digits=1))%");
    println("  ACF-MAE:       $(round(m.acf_mae, digits=4))");
    println("  Kurtosis:      $(round(m.kurtosis, digits=2)) (obs: $(round(m.kurt_obs, digits=2)))");
    println("  Wasserstein-1: $(round(m.wasserstein, digits=3))");
    println("  Hellinger:     $(round(m.hellinger, digits=4))");
    println("  EM iterations: $(length(model_k.log_likelihood_history))");
end

## Summary Table T1: Sensitivity Results

In [ ]:
# --- TABLE T1: STATE-RESOLUTION SENSITIVITY ---
println("\n" * "="^80);
println("Table T1: State-Resolution Sensitivity — $(primary_ticker) (CHMM, no jumps)");
println("="^80);
println(rpad("K", 6) *
        rpad("KS (%)", 10) *
        rpad("Kurt (sim)", 12) *
        rpad("Kurt (obs)", 12) *
        rpad("ACF-MAE", 10) *
        rpad("W-1", 10) *
        rpad("Hellinger", 10));
println("-"^80);

for K in K_VALUES
    m = sensitivity_results[K];
    println(
        rpad(K, 6) *
        rpad(round(m.ks_rate, digits=1), 10) *
        rpad(round(m.kurtosis, digits=2), 12) *
        rpad(round(m.kurt_obs, digits=2), 12) *
        rpad(round(m.acf_mae, digits=4), 10) *
        rpad(round(m.wasserstein, digits=3), 10) *
        rpad(round(m.hellinger, digits=4), 10)
    );
end
println("="^80);
println("Paths per K: $(N_PATHS) | Path length: $(number_of_steps) | MAX_ITER: $(MAX_ITER)");

## Sensitivity Plots

In [ ]:
# --- FIGURE: KS PASS RATE vs K ---
ks_rates = [sensitivity_results[K].ks_rate for K in K_VALUES];

p_ks = plot(K_VALUES, ks_rates, lw=2, color=:navy, marker=:circle, ms=6,
    title="(a) KS Pass Rate vs Number of States K", titlefontsize=10,
    xlabel="K (number of hidden states)", ylabel="KS Pass Rate (%)",
    legend=false, ylims=(0, 105));
hline!(p_ks, [95.0], lw=1, color=:red, ls=:dash, label="95% target");

display(p_ks);

In [ ]:
# --- FIGURE: ACF-MAE vs K ---
acf_maes = [sensitivity_results[K].acf_mae for K in K_VALUES];

p_acf = plot(K_VALUES, acf_maes, lw=2, color=:darkgreen, marker=:diamond, ms=6,
    title="(b) ACF-MAE vs Number of States K", titlefontsize=10,
    xlabel="K (number of hidden states)", ylabel="ACF-MAE",
    legend=false);

display(p_acf);

In [ ]:
# --- FIGURE: KURTOSIS (observed vs simulated) vs K ---
kurt_sim_vals = [sensitivity_results[K].kurtosis for K in K_VALUES];
kurt_obs_val = sensitivity_results[K_VALUES[1]].kurt_obs;  # same for all K

p_kurt = plot(K_VALUES, kurt_sim_vals, lw=2, color=:purple, marker=:square, ms=6,
    title="(c) Excess Kurtosis vs Number of States K", titlefontsize=10,
    xlabel="K (number of hidden states)", ylabel="Excess Kurtosis",
    label="Simulated (mean)");
hline!(p_kurt, [kurt_obs_val], lw=2, color=:red, ls=:dash, label="Observed");

display(p_kurt);

In [ ]:
# --- COMBINED SENSITIVITY FIGURE ---
fig_sens = plot(p_ks, p_acf, p_kurt, layout=(1, 3), size=(1400, 400),
    plot_title="State-Resolution Sensitivity — $(primary_ticker) (CHMM)",
    plot_titlefontsize=12);
display(fig_sens);

savefig(fig_sens, joinpath(_PATH_TO_FIGURES, "Fig-Sensitivity-K-$(primary_ticker).svg"));
savefig(fig_sens, joinpath(_PATH_TO_FIGURES, "Fig-Sensitivity-K-$(primary_ticker).pdf"));

## Cross-Asset Validation

Train CHMM at K=6 on each ticker and compare metrics.
Missing tickers are skipped gracefully.

In [ ]:
# --- CROSS-ASSET CONFIGURATION ---
cross_tickers = ["SPY", "NVDA", "JNJ", "JPM"];
cross_K = 6;
cross_results = Dict{String, NamedTuple}();

In [ ]:
# --- CROSS-ASSET LOOP ---
for tkr in cross_tickers
    println("\n" * "="^50);
    println("Processing $tkr (K=$cross_K)...");
    println("="^50);

    # Check if ticker exists in dataset
    if !haskey(dataset, tkr)
        println("  WARNING: $tkr not found in dataset. Skipping.");
        continue;
    end

    # Compute returns for this ticker
    idx_tkr = findfirst(x -> x == tkr, list_of_all_tickers);
    if isnothing(idx_tkr)
        println("  WARNING: $tkr not in ticker list. Skipping.");
        continue;
    end
    R_tkr = all_firms_excess_return_matrix[:, idx_tkr];
    n_steps_tkr = length(R_tkr);

    # Train CHMM
    try
        model_tkr = build(MyContinuousHiddenMarkovModel, (
            observations = R_tkr,
            number_of_states = cross_K,
            max_iter = MAX_ITER
        ));

        # Stationary distribution
        T_tkr = zeros(cross_K, cross_K);
        for i in 1:cross_K; T_tkr[i, :] = probs(model_tkr.transition[i]); end
        π_tkr = (T_tkr^1000)[1, :];
        start_dist_tkr = Categorical(π_tkr);

        # Simulate
        decoded_tkr = Array{Float64,2}(undef, n_steps_tkr, N_PATHS);
        for p in 1:N_PATHS
            start_state = rand(start_dist_tkr);
            states = model_tkr(start_state, n_steps_tkr);
            for j in 1:n_steps_tkr
                decoded_tkr[j, p] = rand(model_tkr.emission[states[j]]);
            end
        end

        # Evaluate
        m = eval_metrics(R_tkr, decoded_tkr; L_acf=L);
        cross_results[tkr] = m;

        println("  KS pass rate:  $(round(m.ks_rate, digits=1))%");
        println("  ACF-MAE:       $(round(m.acf_mae, digits=4))");
        println("  Kurtosis:      $(round(m.kurtosis, digits=2)) (obs: $(round(m.kurt_obs, digits=2)))");
        println("  Wasserstein-1: $(round(m.wasserstein, digits=3))");
        println("  Hellinger:     $(round(m.hellinger, digits=4))");

    catch e
        println("  ERROR processing $tkr: $e");
        continue;
    end
end

## Cross-Asset Summary Table

In [ ]:
# --- TABLE T2: CROSS-ASSET COMPARISON ---
println("\n" * "="^80);
println("Table T2: Cross-Asset Validation — CHMM (K=$cross_K, no jumps)");
println("="^80);
println(rpad("Ticker", 10) *
        rpad("KS (%)", 10) *
        rpad("Kurt (sim)", 12) *
        rpad("Kurt (obs)", 12) *
        rpad("ACF-MAE", 10) *
        rpad("W-1", 10) *
        rpad("Hellinger", 10));
println("-"^80);

for tkr in cross_tickers
    if !haskey(cross_results, tkr)
        println(rpad(tkr, 10) * "--- skipped ---");
        continue;
    end
    m = cross_results[tkr];
    println(
        rpad(tkr, 10) *
        rpad(round(m.ks_rate, digits=1), 10) *
        rpad(round(m.kurtosis, digits=2), 12) *
        rpad(round(m.kurt_obs, digits=2), 12) *
        rpad(round(m.acf_mae, digits=4), 10) *
        rpad(round(m.wasserstein, digits=3), 10) *
        rpad(round(m.hellinger, digits=4), 10)
    );
end
println("="^80);
println("Paths: $(N_PATHS) | MAX_ITER: $(MAX_ITER)");

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice.